# Feature Engineering - Cervical Cancer Risk Factors

This notebook handles data preprocessing, missing value imputation, feature scaling, and prepares the data for model training.

In [3]:
%pip install imblearn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 258 kB 4.4 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from utils import (load_raw_data, convert_to_numeric, missing_value_summary,
                   drop_high_missing_columns, impute_missing, drop_redundant_columns,
                   setup_notebook, TARGET)

setup_notebook()
print('Libraries imported successfully.')

In [ ]:
# Load the dataset ('?' placeholders are replaced with NaN)
df = load_raw_data()
print(f'Original Dataset Shape: {df.shape}')
df.head()

## 1. Data Type Conversion

Convert all columns to numeric (currently some are stored as object/string due to '?' values)

In [ ]:
# Convert all columns to numeric
convert_to_numeric(df)
print('All columns converted to numeric.')
print(f'Data types after conversion:\n{df.dtypes.value_counts()}')

## 2. Missing Value Analysis & Imputation Strategy

In [ ]:
# Display missing value counts
missing_info = missing_value_summary(df)
print('Missing Values Summary:')
missing_info

In [ ]:
# Drop columns with very high missing values (>80%)
high_missing_cols = drop_high_missing_columns(df, threshold=80)
print(f'Dropped columns with >80% missing values: {high_missing_cols}')
print(f'Shape after dropping high-missing columns: {df.shape}')

In [ ]:
# Imputation Strategy (see utils.impute_missing):
# - Binary columns: mode (most frequent value)
# - Continuous columns: mean
# - Count-like columns: median
impute_missing(df)

print(f'Shape after imputation: {df.shape}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

## 3. Drop Redundant Columns

In [ ]:
# Drop redundant columns that have separate year/pack info
existing_drop_cols = drop_redundant_columns(df)
print(f'Dropped columns: {existing_drop_cols}')
print(f'Shape after dropping redundant columns: {df.shape}')

## 4. Remove Duplicates

In [11]:
# Check and remove duplicates
print(f'Duplicate rows before removal: {df.duplicated().sum()}')
df.drop_duplicates(keep='first', inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

Duplicate rows before removal: 24
Shape after removing duplicates: (834, 30)


## 5. Feature Scaling & Train-Test Split

In [ ]:
# Separate features and target
X = df.drop(TARGET, axis=1)
y = df[TARGET]

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nTarget distribution:\n{y.value_counts()}')

In [13]:
# Train-test split (stratified to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'\nTrain target distribution:\n{y_train.value_counts()}')

X_train shape: (667, 29)
X_test shape: (167, 29)

Train target distribution:
Biopsy
0    624
1     43
Name: count, dtype: int64


In [14]:
# Feature scaling using StandardScaler
scaler = StandardScaler()

# Fit on training data only, transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print('Feature scaling completed.')
print(f'\nScaled training data statistics:')
X_train_scaled.describe()

Feature scaling completed.

Scaled training data statistics:


,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes (years),Smokes (packs/year),Hormonal Contraceptives (years),IUD (years),STDs (number),STDs:condylomatosis,...,STDs:Hepatitis B,STDs:HPV,STDs: Number of diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology
count,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,...,667.000000,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02,6.670000e+02
mean,1.038649e-16,-1.398182e-16,5.639334e-16,-7.723291e-17,5.326407e-18,1.065281e-17,1.464762e-17,-5.326407e-18,-6.391689e-17,-3.195844e-17,...,0.000000,5.326407e-18,2.130563e-17,3.728485e-17,1.864243e-17,-3.994805e-17,7.456970e-17,1.331602e-17,-4.926927e-17,-1.597922e-17
std,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,...,1.000750,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00,1.000750e+00
min,-1.633051e+00,-8.972423e-01,-2.467772e+00,-1.629516e+00,-3.082281e-01,-2.095661e-01,-6.527376e-01,-2.840294e-01,-2.924305e-01,-2.317595e-01,...,-0.038749,-5.484085e-02,-2.860507e-01,-1.464224e-01,-1.169522e-01,-1.409882e-01,-1.712337e-01,-2.132007e-01,-3.143991e-01,-2.388562e-01
25%,-7.013754e-01,-3.190195e-01,-7.442009e-01,-9.000617e-01,-3.082281e-01,-2.095661e-01,-6.527376e-01,-2.840294e-01,-2.924305e-01,-2.317595e-01,...,-0.038749,-5.484085e-02,-2.860507e-01,-1.464224e-01,-1.169522e-01,-1.409882e-01,-1.712337e-01,-2.132007e-01,-3.143991e-01,-2.388562e-01
50%,-1.190784e-01,-3.190195e-01,-5.477245e-02,-1.706071e-01,-3.082281e-01,-2.095661e-01,-3.654891e-01,-2.840294e-01,-2.924305e-01,-2.317595e-01,...,-0.038749,-5.484085e-02,-2.860507e-01,-1.464224e-01,-1.169522e-01,-1.409882e-01,-1.712337e-01,-2.132007e-01,-3.143991e-01,-2.388562e-01
75%,5.796780e-01,2.592033e-01,2.899418e-01,5.588475e-01,-3.082281e-01,-2.095661e-01,1.371957e-01,-2.840294e-01,-2.924305e-01,-2.317595e-01,...,-0.038749,-5.484085e-02,-2.860507e-01,-1.464224e-01,-1.169522e-01,-1.409882e-01,-1.712337e-01,-2.132007e-01,-3.143991e-01,-2.388562e-01
max,6.635567e+00,1.471477e+01,5.115941e+00,6.394485e+00,8.140603e+00,1.518204e+01,7.964717e+00,1.026113e+01,7.356634e+00,4.314817e+00,...,25.806976,1.823458e+01,9.415432e+00,6.829558e+00,8.550504e+00,7.092792e+00,5.839971e+00,4.690416e+00,3.180671e+00,4.186619e+00


## 6. Handling Class Imbalance with SMOTE

In [15]:
# Apply SMOTE to handle class imbalance
print(f'Before SMOTE:')
print(f'Class 0: {sum(y_train == 0)}')
print(f'Class 1: {sum(y_train == 1)}')

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'\nAfter SMOTE:')
print(f'Class 0: {sum(y_train_resampled == 0)}')
print(f'Class 1: {sum(y_train_resampled == 1)}')
print(f'\nResampled training shape: {X_train_resampled.shape}')

Before SMOTE:
Class 0: 624
Class 1: 43

After SMOTE:
Class 0: 624
Class 1: 624

Resampled training shape: (1248, 29)


## 7. Save Processed Data for Model Training

In [ ]:
# Save processed datasets
np.save('X_train_resampled.npy', X_train_resampled)
np.save('X_test_scaled.npy', X_test_scaled)
np.save('y_train_resampled.npy', y_train_resampled)
np.save('y_test.npy', y_test)

# Save column names for reference
with open('feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)

print('Processed datasets saved successfully.')
print(f'\nFiles created:')
print(f'  - X_train_resampled.npy: {X_train_resampled.shape}')
print(f'  - X_test_scaled.npy: {X_test_scaled.shape}')
print(f'  - y_train_resampled.npy: {y_train_resampled.shape}')
print(f'  - y_test.npy: {y_test.shape}')
print(f'  - feature_columns.json: {len(X.columns)} features')

## 8. Feature Engineering Summary

In [17]:
print('=' * 70)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 70)
print()
print('1. Data Loading: Loaded CSV with {} rows'.format(858))
print()
print('2. Missing Value Handling:')
print('   - Replaced "?" with NaN')
print(f'   - Dropped high-missing columns: {high_missing_cols}')
print('   - Binary columns: mode imputation')
print('   - Continuous columns: mean imputation')
print('   - Count columns: median imputation')
print()
print('3. Dropped Redundant Columns:')
print(f'   - {existing_drop_cols}')
print()
print('4. Duplicates Removed')
print()
print('5. Train-Test Split: 80/20 (stratified)')
print()
print('6. Feature Scaling: StandardScaler')
print()
print('7. Class Imbalance: SMOTE applied to training data')
print()
print(f'8. Final Training Set: {X_train_resampled.shape[0]} samples with {X_train_resampled.shape[1]} features')
print(f'   Final Test Set: {X_test_scaled.shape[0]} samples with {X_test_scaled.shape[1]} features')
print()
print('9. Next Step: Model Training and Evaluation')

FEATURE ENGINEERING SUMMARY

1. Data Loading: Loaded CSV with 858 rows

2. Missing Value Handling:
   - Replaced "?" with NaN
   - Dropped high-missing columns: ['STDs: Time since last diagnosis', 'STDs: Time since first diagnosis']
   - Binary columns: mode imputation
   - Continuous columns: mean imputation
   - Count columns: median imputation

3. Dropped Redundant Columns:
   - ['Smokes', 'STDs', 'Hormonal Contraceptives', 'IUD']

4. Duplicates Removed

5. Train-Test Split: 80/20 (stratified)

6. Feature Scaling: StandardScaler

7. Class Imbalance: SMOTE applied to training data

8. Final Training Set: 1248 samples with 29 features
   Final Test Set: 167 samples with 29 features

9. Next Step: Model Training and Evaluation
